<a href="https://colab.research.google.com/github/abhisheknaik2k20/SMA_Project/blob/main/SMA_Experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Import/Install Packages**

In [1]:
!pip install supabase
from supabase import Client, create_client
from google.colab import userdata as env
from datetime import datetime
from googleapiclient.discovery import build
import requests as req
import json

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 730.0/730.0 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 5.3 MB/s eta 0:00:00
  Attempting uninstall: cachetools
    Found existing installation: cachetools 7.0.1
    Uninstalling cachetools-7.0.1:
      Successfully uninstalled cachetools-7.0.1


# **POSTGRE DATABASE API CODE**

In [34]:
class PostgreSQLDatabase:
  def __init__(self):
    try:
      self.supabase_client : Client = create_client(env.get('supabase_URL'), env.get('supabase_key'))
    except Exception as error : print(error)

  def populate_genre(self, data=[]):
    try:
      response=self.supabase_client.table('Genre').insert(data, count='exact').execute()
      return response.data, response.count
    except Exception as error : print(error)

  def get_genre_info(self):
    try:
      response=self.supabase_client.table('Genre').select('*', count ='exact').execute()
      return response.data, response.count
    except Exception as error : print(error)

  def populate_video(self, data=[]):
    try:
      response = self.supabase_client.table('Video').insert(data, count='exact').execute()
      return response.data, response.count
    except Exception as error : print(error)

  def get_video_info(self):
    try:
      response=self.supabase_client.table('Video').select('*', count ='exact').execute()
      return response.data, response.count
    except Exception as error : print(error)

  def return_video_ids(self):
    try:
      response= self.supabase_client.table('Video').select('video_id').execute()
      return [r['video_id'] for r in response.data]
    except Exception as error : print(error)

  def populate_comments(self, data=[]):
    try:
      response = self.supabase_client.table('Comments').insert(data, count='exact').execute()
      return response.data, response.count
    except Exception as error : print(error)

  def fetch_comments_data(self):
    try:
      response=self.supabase_client.table('Comments').select('*').execute()
      return response.data
    except Exception as error : print(error)

  def return_comment_ids(self):
    try:
      response= self.supabase_client.table('Comments').select('comment_id').execute()
      return [r['comment_id'] for r in response.data]
    except Exception as error : print(error)

  def return_video_ids_with_no_comment_data(self):
    try:
      video_ids = set(self.return_video_ids())
      commented_video_ids = set(r['video_id'] for r in self.supabase_client.rpc('get_distinct_comment_video_ids').execute().data)
      return list(video_ids.difference(commented_video_ids))
    except Exception as error : print(error)

  def get_channel_ids(self):
    try:
      channel_ids = [r['channel_id'] for r in self.supabase_client.rpc('get_distinct_channel_ids').execute().data]
      return channel_ids
    except Exception as error : print(error)

  def populate_channel(self, channel_data=[]):
    try : return self.supabase_client.table('Channel').insert(channel_data, count='exact').execute()
    except Exception as error : print(error)

  def populate_subreddit(self, subreddits = []):
    try : return self.supabase_client.table('Subreddits').insert(subreddits, count='exact').execute()
    except Exception as error : print(error)

  def populate_posts(self, posts = []):
    try : return self.supabase_client.table('Posts').insert(posts, count='exact').execute()
    except Exception as error : print(error)

# **Reddit API Code**

In [66]:
class FetchRedditData:
  def __init__(self):
    self.url = env.get('reddit_API_URL')
    self.headers = {'x-rapidapi-key': env.get('reddit_API_key'),'x-rapidapi-host': env.get('reddit_API_host')}

  def make_api_call(self, endpoint='', query_params={}):
    try:
      response = req.get(f"{self.url}/{endpoint}", headers=self.headers, params=query_params)
      return response.json()
    except Exception as error : print(error)

  def return_popular_subreddits(self):
    response=self.make_api_call(endpoint='getPopularSubreddits')
    if response['success'] :
      print(response)
      result=[]
      for sub_data in response['data']['subreddits']:
        data = sub_data['data']
        result.append({'id' : data.get('id'),'name' : data.get('display_name'),'name_prefixed' : data.get('display_name_prefixed'),'title' : data.get('title'),'url' : data.get('url'),'icon_url' : data.get('icon_img'),'banner_url' : data.get('banner_img'),'header_url' : data.get('header_img'),'subscribers' : data.get('subscribers'),'public_description' : data.get('public_description'),'description_html' : data.get('description_html'),'description' : data.get('description'),'created_utc' : datetime.fromtimestamp(data.get('created_utc')).isoformat(),'primary_color' : data.get('primary_color'),'key_color' : data.get('key_color'),'banner_background_color' : data.get('banner_background_color'),'advertiser_category' : data.get('advertiser_category')})
      return result

  def store_popular_subreddits(self):
    pgdb = PostgreSQLDatabase()
    subreddits = self.return_popular_subreddits()
    if not subreddits : return
    pgdb.populate_subreddit(subreddits=subreddits)

  def return_popular_posts_from_subreddit(self, query_params={}):
    return self.make_api_call(endpoint='getTopPostsBySubreddit', query_params= query_params)

  def store_popular_posts_from_subreddit(self, ):
    pgdb = PostgreSQLDatabase()
    subreddits = pgdb.supabase_client.table('Subreddits').select('*').execute().data
    for subreddit in subreddits:
      response = reddit.return_popular_posts_from_subreddit(query_params= {"subreddit":subreddit['name'],"time":"year"})
      result = []
      for sub_data in response['data']['posts']:
        data = sub_data['data']
        result.append({"id": data.get('id'),"name": data.get('name'),"permalink": data.get('permalink'),"title": data.get('title'),"author": data.get('author'),"subreddit_id": data.get('subreddit_id')[3:],"created_utc": datetime.fromtimestamp(data.get('created_utc', 0)).isoformat(),"score": data.get('score'),"ups": data.get('ups'),"upvote_ratio": data.get('upvote_ratio'),"num_comments": data.get('num_comments'),"num_crossposts": data.get('num_crossposts'),"total_awards_received": data.get('total_awards_received'),"domain": data.get('domain'),"url": data.get('url'),"post_hint": data.get('post_hint'),"is_video": data.get('is_video'),"is_gallery": data.get('is_gallery'),"thumbnail_url": data.get('thumbnail'),"preview" : data.get('preview')})
      pgdb.populate_posts(result)

In [67]:
reddit = FetchRedditData()
reddit.store_popular_posts_from_subreddit()

# **YOUTUBE API CODE**

In [2]:
class FetchYoutubeData:
  def __init__(self):
    try : self.youtube = build("youtube", "v3", developerKey=env.get('YT_v33') )
    except Exception as error :
      print(error)

  def fetch_video_ids(self, videoCategoryTag : int, order = 'viewCount'):
    try:
      search_response = self.youtube.search().list(part="id",type="video",videoCategoryId=videoCategoryTag, q=" ", order=order,relevanceLanguage="en",maxResults=20).execute()
      return [item["id"]["videoId"] for item in search_response["items"]]
    except Exception as error :
      print(error)

  def fetch_video_data(self,genre_id,video_ids):
    video_data=[]
    for video_id in video_ids:
      try:
        video_request = self.youtube.videos().list(part="snippet,contentDetails,statistics,status,topicDetails", id =  video_id).execute()
        item=video_request["items"][0]
        video_data.append({'video_id': item['id'],'title': item['snippet']['title'],'description': item['snippet']['description'],'published_at': item['snippet']['publishedAt'],'channel_id': item['snippet']['channelId'],'channel_title':item['snippet']['channelTitle'],'comment_count': int(item['statistics'].get('commentCount', 0)),'like_count': int(item['statistics'].get('likeCount', 0)),'view_count': int(item['statistics'].get('viewCount', 0)),'genre_id': genre_id})
      except Exception as error : continue
    self.fetch_channel_data(channel_ids= [r['channel_id'] for r in video_data])
    return video_data

  def store_video_data(self):
    pgdb = PostgreSQLDatabase()
    responses, count = pgdb.get_genre_info()
    if count == -1 : return
    stored_video_data = set(pgdb.return_video_ids())
    for response in responses:
        video_ids = self.fetch_video_ids(response['video_tag'], order = 'relevance')
        if not video_ids : continue
        result = set(video_ids).difference(stored_video_data)
        if not result: continue
        stored_video_data.update(result)
        pgdb.populate_video(self.fetch_video_data(response['genre_id'], result))

  def fetch_comment_data(self, video_id):
    result=[]
    try:
      request = self.youtube.commentThreads().list(part="snippet",videoId=video_id,textFormat="plainText", order="relevance", maxResults=10).execute()
      for item in request.get("items", []):
        top_comment = item["snippet"]["topLevelComment"]["snippet"]
        result.append({"comment_id": item["id"],"video_id": video_id,"comment_text": top_comment["textDisplay"],"like_count": str(top_comment["likeCount"]),"comment_date": top_comment["publishedAt"][:10]})
      return result
    except Exception as error : return []

  def fetch_channel_data(self, channel_ids=[]):
    existing_channel_data = set(PostgreSQLDatabase().get_channel_ids())
    channel_data=[]
    for channel_id in channel_ids:
        try:
            response = self.youtube.channels().list(part="snippet,statistics,contentDetails,localizations,topicDetails",id=channel_id).execute()
            item = response['items'][0]
            if item['id'] not in existing_channel_data :
              channel_data.append({'channel_id': item['id'],'channel_title': item['snippet']['title'],'description': item['snippet']['description'],'published_at': item['snippet']['publishedAt'],'country': item['snippet'].get('country', None),'default_language': item['snippet'].get('defaultLanguage', None),'view_count': int(item['statistics'].get('viewCount', 0)),'subscriber_count': int(item['statistics'].get('subscriberCount', 0)),'video_count': int(item['statistics'].get('videoCount', 0)),'topic_categories': item.get('topicDetails', {}).get('topicCategories', [])})
        except Exception as error:continue
    PostgreSQLDatabase().populate_channel(channel_data=channel_data)


  def store_comment_data(self):
     pgdb = PostgreSQLDatabase()
     video_ids = pgdb.return_video_ids_with_no_comment_data()
     for video_id in video_ids:
      comments = self.fetch_comment_data(video_id)
      if not comments : continue
      pgdb.populate_comments(comments)

**Fetching and Storing Youtube Video Data**

In [122]:
youtube = FetchYoutubeData()
youtube.store_video_data()
youtube.store_comment_data()

In [10]:
reddit = FetchRedditData()
reddit.return_popular_posts_from_subreddit()

[{'name': 'Home'}, {'name': 'AskReddit'}, {'name': 'NoStupidQuestions'}, {'name': 'BaldursGate3'}, {'name': 'facepalm'}, {'name': 'interestingasfuck'}, {'name': 'Damnthatsinteresting'}, {'name': 'Helldivers'}, {'name': 'LivestreamFail'}, {'name': 'pics'}, {'name': 'Palworld'}, {'name': 'mildlyinfuriating'}, {'name': 'Piracy'}, {'name': 'PeterExplainsTheJoke'}, {'name': 'funny'}, {'name': 'AITAH'}, {'name': 'movies'}, {'name': 'gaming'}, {'name': 'worldnews'}, {'name': 'leagueoflegends'}, {'name': 'pcmasterrace'}, {'name': 'Unexpected'}, {'name': 'news'}, {'name': 'politics'}]
